# HW09 — Основы VLM / Vision-language models
**CIFAR-10:** [источник / source](https://www.cs.toronto.edu/~kriz/cifar.html). Одна строка — реальное RGB-изображение 32×32; пиксели доступны при запросе, целевая метка — одна из 10 категорий. / Each row is a real 32×32 RGB image; pixels are available at request time, and the target is one of ten categories.

**Модель / Model:** `openai/clip-vit-base-patch32`, revision `3d74acf9a28c67741b2f4f2ea7635f0aaf6f0268`. Замороженные признаки заранее вычислены из реальных изображений официальным CLIPProcessor; это не синтетика и не генератор случайных признаков. Ноутбук работает на CPU без загрузки весов. Скрипт пересчёта и SHA256 опубликованы рядом с данными. / Frozen features were computed from real images using the official CLIPProcessor; they are not synthetic or random features. This CPU notebook requires no model download. The recomputation script and SHA256 hashes accompany the data.

**Разбиение / Splits:** 100 Validation из official train и 100 Test из official test, по 10 каждого класса. Практика использует первые 10 объектов класса, ДЗ — следующие 10: их ID не пересекаются. Train для обучения головы не нужен: выбираем лишь формулировку текста по Validation. Test открываем один раз после выбора. / 100 validation observations from official train and 100 test observations from official test, ten per class. Practice uses the first ten of each class; homework uses the next ten, with disjoint IDs. No head-training split is required: only wording is selected on validation. Open Test once after selection.

**Ограничения / Limitations:** маленькая непредставительная выборка; нет гарантий разделения по автору/сцене; пересечение с предобучением неизвестно. Это не полный benchmark. / Small nonrepresentative subset; no guaranteed author/scene separation; pretraining overlap is unknown. This is not a full benchmark.

**Формат / Format:** ровно четыре блока: материал → пример → задание → решение → проверка. Colab содержит NumPy, sklearn и matplotlib. / Exactly four blocks: material → example → task → solution → check. Colab includes NumPy, sklearn, and matplotlib.

**Оценивание / Grading:** код 80 (20 за блок), анализ 20 (5 за блок). Срок — живая матрица группы; линейный штраф за 14 суток после срока; отсутствие P09 даёт ×0.9 после просрочки. / Code 80 (20 per block), analysis 20 (5 per block). Deadline: live group matrix; linear deduction over 14 days after the deadline; missing P09 adds ×0.9 after lateness.


In [ ]:
STUDENT_NAME = '' # Введите подтверждённое ФИО / Enter your confirmed full name.
STUDENT_GROUP = '' # Введите группу / Enter your group.

In [ ]:
import json, hashlib, urllib.request # Читать манифест, проверять SHA256 и скачивать данные / Read provenance, verify SHA256, and download data.
from pathlib import Path # Работать с локальным кешем / Manage a local cache.
import numpy as np # Вычислять сходства в массивах / Compute similarities in arrays.
import pandas as pd # Показывать таблицы метрик / Display metric tables.
import matplotlib.pyplot as plt # Показывать изображения и матрицу / Display images and a confusion matrix.
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay # Эталонные метрики для проверок / Reference metrics for checks.
CACHE = Path('vlm_data') # Отдельная папка учебных данных / Dedicated teaching-data directory.
CACHE.mkdir(exist_ok=True) # Создать кеш при первом запуске / Create the cache on the first run.
BASE = 'https://raw.githubusercontent.com/alexander-toschev/ml-cs-intro/main/data/09-vlm/' # Официальный путь материалов курса / Course material URL.
KIND = 'homework' # Использовать отдельные данные задания / Use assignment-specific data.
HASHES = {'homework_validation.npz': '2c8f978bde27ca30ce4b30f4bbb38280bc731b290071eaf9548f008e5cade309', 'practice_validation.npz': 'f533117fcd463a717435364b46ac10a5673fe0789ebf3a767cf94b4391668ec1', 'homework_test.npz': 'ba0d27f266aea143ce1ae6c7d657b1925da8902c09320813d336a3249007ca50', 'practice_test.npz': 'bbf9e5a2edd77b08b347ae16749bb88b3d4866481d575b13255df87f3204cc9e'} # Проверять целостность каждого файла / Verify each file’s integrity.
MODEL_REV = '3d74acf9a28c67741b2f4f2ea7635f0aaf6f0268' # Зафиксировать происхождение признаков / Pin feature provenance.
def load_part(split): # Загружать только явно запрошенную часть / Load only the explicitly requested partition.
    name = KIND + '_' + split + '.npz' # Связать тип задания и роль выборки / Combine assignment kind and partition role.
    path = CACHE / name # Выбрать локальный файл кеша / Select the local cache file.
    if not path.exists(): # Не скачивать повторно / Avoid repeated downloads.
        urllib.request.urlretrieve(BASE + name, path) # Загрузить опубликованные реальные признаки / Download published real-data features.
    assert hashlib.sha256(path.read_bytes()).hexdigest() == HASHES[name] # Отвергнуть изменённые данные / Reject modified data.
    with np.load(path, allow_pickle=False) as archive: # Не исполнять сериализованные объекты / Disable object deserialization.
        result = {key: archive[key] for key in archive.files} # Скопировать числовые массивы и строки / Copy numerical arrays and strings.
    assert len(set(result['ids'])) == len(result['ids']) # Проверить уникальные исходные ID / Verify unique source IDs.
    return result # Вернуть признаки, изображения, метки и описания / Return features, images, labels, and descriptions.
val = load_part('validation') # Открыть только Validation до выбора / Open validation only before selection.
classes = val['classes'].tolist() # Сохранить исходный порядок десяти классов / Retain the original order of ten classes.
print({key: value.shape for key, value in val.items()}) # Показать формы для проверки контракта / Display shapes to inspect the contract.

## 1. Нормализация / Normalization

Косинусное сходство зависит от направления вектора. Нормализуйте каждую строку по L2-норме; нулевую строку оставьте нулевой. / Cosine similarity depends on direction. Normalize each row by its L2 norm; keep zero rows at zero.

In [ ]:
one = val['features'][0] # Взять реальный вектор изображения / Take a real image vector.
print(np.linalg.norm(one), np.linalg.norm(one / np.linalg.norm(one))) # Сравнить длину до и после нормализации / Compare length before and after normalization.

**Задание / Task**

Напишите `normalize_rows(values)` для массива N×D. Верните новый float-массив той же формы, вход не меняйте. Ненулевые строки должны иметь норму 1, нулевые — остаться нулевыми. Проверьте масштабирование реальных признаков. / Write `normalize_rows(values)` for an N×D array. Return a new floating-point array of the same shape without changing the input. Nonzero rows must have unit norm; zero rows stay zero. Verify scaling on real features.

In [ ]:
def normalize_rows(values): # Реализуйте контракт задания / Implement the exercise contract.
    raise NotImplementedError("Complete exercise 1") # Замените своим кодом / Replace with your code.

In [ ]:
original = val['features'].copy() # Сохранить реальные признаки для проверки неизменности / Save real features to check immutability.
unit = normalize_rows(original) # Нормализовать признаки Validation / Normalize validation features.
assert unit.shape == original.shape and np.isfinite(unit).all() # Проверить форму и конечность / Check shape and finite values.
assert np.allclose(np.linalg.norm(unit, axis=1), 1) # Проверить единичную длину реальных ненулевых векторов / Verify unit length for real nonzero vectors.
assert np.array_equal(original, val['features']) # Проверить отсутствие изменения входа / Verify that input was not modified.
assert np.allclose(normalize_rows(original * 3), unit) # Масштаб не должен менять направление / Scaling must not change direction.
zero = original[:1] - original[:1] # Получить граничный нулевой вектор из реальных данных / Derive a boundary-case zero vector from real data.
assert np.array_equal(normalize_rows(zero), zero) # Проверить оговорённое поведение на нуле / Verify the specified zero-vector behavior.
print('Exercise 1: OK / Задание 1: OK') # Сообщить о пройденных проверках / Report successful checks.

**Ваш вывод / Your explanation:** ...

## 2. Сходства и прогнозы / Similarities and predictions

У каждой категории есть текстовый вектор. Матрица N×10 содержит сходство каждого изображения с каждым текстом. / Each category has a text vector. The N×10 matrix contains each image’s similarity to every description.

In [ ]:
example_score = normalize_rows(val['features'][:1]) @ normalize_rows(val['text_bare'][:1]).T # Сравнить одно изображение с первым текстом / Compare one image with the first description.
print(example_score) # Показать сходство, не вероятность / Display a similarity, not a probability.

**Задание / Task**

Напишите `classify(image_features, text_features)`. Нормализуйте обе матрицы, рассчитайте косинусные сходства и верните `(scores, predicted)`. Столбцы должны соответствовать порядку текстов. При равенстве выберите первый столбец. Не используйте истинные метки. / Implement `classify(image_features, text_features)`. Normalize both matrices and return cosine `(scores, predicted)`. Columns follow text ordering; choose the first column on ties. Do not use reference labels.

In [ ]:
def classify(image_features, text_features): # Реализуйте контракт задания / Implement the exercise contract.
    raise NotImplementedError("Complete exercise 2") # Замените своим кодом / Replace with your code.

In [ ]:
bare_scores, bare_pred = classify(val['features'], val['text_bare']) # Получить baseline на реальных изображениях / Compute the baseline on real images.
reference = normalize_rows(val['features']) @ normalize_rows(val['text_bare']).T # Независимо собрать эталонное произведение / Independently construct the reference product.
assert bare_scores.shape == (len(val['labels']), 10) # Проверить число изображений и кандидатов / Check image and candidate counts.
assert np.allclose(bare_scores, reference) and np.array_equal(bare_pred, reference.argmax(axis=1)) # Сверить сходства и прогнозы / Verify similarities and predictions.
reversed_scores, reversed_pred = classify(val['features'], val['text_bare'][::-1]) # Изменить порядок текстов без изменения изображений / Reverse text order while retaining images.
assert np.allclose(reversed_scores, bare_scores[:, ::-1]) # Столбцы должны переставляться вместе с текстами / Columns must follow text ordering.
print('Exercise 2: OK / Задание 2: OK') # Подтвердить корректность классификации / Confirm correct classification.

**Ваш вывод / Your explanation:** ...

## 3. Контролируемое сравнение / Controlled comparison

До запуска запишите прогноз: поможет ли фраза “a photo of a ...”? Не меняйте картинки, веса или метрику. / Before running the candidate, predict whether “a photo of a ...” will help. Keep images, weights, and metric fixed.

In [ ]:
print(classes[0], '→', 'a photo of a ' + classes[0]) # Показать единственное изменение конфигурации / Show the only configuration change.

**Задание / Task**

Напишите `choose_variant(features, labels, texts)`: варианты `bare` и `photo`. Верните `(selected, reports)`, где для каждого варианта есть accuracy и macro_f1 по всем 10 классам. Выбирайте photo только при строго лучшем macro-F1. При равенстве оставьте bare. После проверки объясните результат в 3–5 предложениях и не открывайте Test раньше сохранения selection. / Implement `choose_variant(features, labels, texts)` for `bare` and `photo`. Return `(selected, reports)` with accuracy and macro_f1 over all ten classes. Select photo only on strict improvement; retain bare on ties. Explain the measured result in 3–5 sentences. Do not open Test before saving selection.

**Прогноз до запуска / Prediction before running:** Запишите здесь / Write here.

In [ ]:
def choose_variant(features, labels, texts): # Реализуйте контракт задания / Implement the exercise contract.
    raise NotImplementedError("Complete exercise 3") # Замените своим кодом / Replace with your code.

In [ ]:
texts = {'bare': val['text_bare'], 'photo': val['text_photo']} # Зафиксировать ровно два варианта текста / Freeze exactly two text variants.
selected, reports = choose_variant(val['features'], val['labels'], texts) # Сравнить варианты только на Validation / Compare variants on validation only.
for name, text_features in texts.items(): # Проверить каждую строку отчёта / Check each report row.
    expected_pred = classify(val['features'], text_features)[1] # Независимо получить соответствующие прогнозы / Independently obtain matching predictions.
    assert np.isclose(reports[name]['macro_f1'], f1_score(val['labels'], expected_pred, labels=np.arange(10), average='macro', zero_division=0)) # Сверить основную метрику / Verify the primary metric.
assert selected == ('photo' if reports['photo']['macro_f1'] > reports['bare']['macro_f1'] else 'bare') # Проверить правило выбора / Verify the selection rule.
assert choose_variant(val['features'], val['labels'], {'bare': texts['bare'], 'photo': texts['bare']})[0] == 'bare' # Проверить ничью на одинаковых реальных текстах / Test ties using identical real descriptions.
display(pd.DataFrame(reports).T) # Показать результат честного сравнения / Display the controlled comparison.
selection = dict(variant=selected, criterion='validation macro-F1; ties bare', validation_ids=val['ids'].tolist(), reports=reports, model_revision=MODEL_REV) # Зафиксировать выбор до открытия Test / Freeze selection before opening Test.
Path('vlm_selection.json').write_text(json.dumps(selection, indent=2)) # Сохранить решение отдельно от финальной оценки / Save selection separately from final evaluation.
print('Exercise 3: OK / Задание 3: OK', selected) # Показать выбранный вариант / Show the selected variant.

**Ваш вывод / Your explanation:** ...

## 4. Ошибки и финальная проверка / Errors and final evaluation

Матрица показывает направления ошибок, а изображения помогают сформулировать гипотезы. / The matrix shows error directions; images help formulate hypotheses.

In [ ]:
print(val['ids'][0], classes[int(val['labels'][0])]) # Связать реальный ID с истинной категорией / Link a real ID to its reference category.

**Задание / Task**

Напишите `error_report(labels, predicted, ids)`: верните матрицу 10×10 (строки true, столбцы predicted) и список всех ошибок в исходном порядке. Каждая запись содержит position, id, true, predicted. После проверки для каждого из показанных примеров отдельно напишите наблюдение и гипотезу; не называйте гипотезу доказанной причиной. Укажите два ограничения обобщения и следующий эксперимент без подбора по Test. Сохраните итоговый журнал и объясните, почему similarity не является вероятностью. / Implement `error_report(labels, predicted, ids)`: return a 10×10 matrix (true rows, predicted columns) and all errors in original order. Each record contains position, id, true, predicted. After the check, write an observation and a hypothesis for each displayed failure; do not present hypotheses as established causes. State two generalization limits and a follow-up without Test tuning. Save the final log and explain why similarity is not probability.

In [ ]:
def error_report(labels, predicted, ids): # Реализуйте контракт задания / Implement the exercise contract.
    raise NotImplementedError("Complete exercise 4") # Замените своим кодом / Replace with your code.

In [ ]:
scores, predicted = classify(val['features'], texts[selected]) # Использовать замороженную формулировку / Use the frozen wording.
matrix, errors = error_report(val['labels'], predicted, val['ids']) # Построить отчёт только по Validation / Build the validation error report.
assert matrix.shape == (10, 10) and matrix.sum() == len(predicted) # Не потерять ни один объект / Account for every observation.
assert np.array_equal(matrix, confusion_matrix(val['labels'], predicted, labels=np.arange(10))) # Проверить ориентацию и значения / Verify orientation and counts.
assert [e['id'] for e in errors] == val['ids'][val['labels'] != predicted].tolist() # Проверить соответствие ошибок исходным ID / Check error IDs against original observations.
assert sum(matrix.diagonal()) + len(errors) == len(predicted) # Сверить правильные ответы и ошибки / Reconcile correct predictions and errors.
ConfusionMatrixDisplay(matrix, display_labels=classes).plot(xticks_rotation=60) # Подписать истинные строки и предсказанные столбцы / Label true rows and predicted columns.
plt.show() # Показать матрицу / Display the matrix.
for error in errors[:3]: # Разобрать до трёх настоящих ошибок / Inspect up to three actual mistakes.
    i = error['position'] # Вернуться к исходной позиции / Recover the original position.
    plt.imshow(val['images'][i]) # Показать исходное изображение 32×32 / Display the original 32×32 image.
    plt.title(error['id'] + ': ' + classes[error['true']] + ' → ' + classes[error['predicted']]) # Связать картинку с метками / Link the image to its labels.
    plt.axis('off') # Убрать координатную сетку фотографии / Hide photograph axes.
    plt.show() # Показать текущую ошибку / Display the current mistake.
    print(sorted(zip(classes, scores[i].tolist()), key=lambda pair: -pair[1])[:2]) # Вывести два наибольших сходства / Print the top two similarities.
if not errors: # Обработать отсутствие наблюдаемых ошибок / Handle the absence of observed errors.
    print('No validation errors / Ошибок на Validation нет') # Не придумывать ошибочные примеры / Do not invent failure examples.
test = load_part('test') # Открыть Test только после сохранённого выбора / Open Test only after saving selection.
assert not set(val['ids']) & set(test['ids']) # Проверить разделение официальных источников / Verify separation of official partitions.
test_scores, test_pred = classify(test['features'], texts[selected]) # Один финальный прогон замороженного варианта / One final evaluation of the frozen variant.
final = dict(accuracy=float(accuracy_score(test['labels'], test_pred)), macro_f1=float(f1_score(test['labels'], test_pred, labels=np.arange(10), average='macro', zero_division=0))) # Оценить все тестовые объекты / Evaluate every test observation.
log = dict(selection=selection, test_ids=test['ids'].tolist(), test_predictions=test_pred.tolist(), test_report=final, validation_errors=errors) # Сохранить воспроизводимые результаты / Save reproducible results.
Path('vlm_experiment.json').write_text(json.dumps(log, ensure_ascii=False, indent=2)) # Записать журнал без выдуманного времени сдачи / Write the log without inventing submission times.
print('Exercise 4: OK / Задание 4: OK', final) # Показать итог без повторного выбора / Display the final result without retuning.

**Ваш вывод / Your explanation:** ...